# Stochastics Is All You Need
## Chasing Determinism in Classical ML, Supervised NLP, and Instruct-tuned Foundation Models

*Simon Reichel*

---
## 1 Environment info

**Distro:** Linux Fedora 43 for Workstation \
**Kernel:** Linux 7.0.12-101.fc43.x86_64 \
**NVIDIA CUDA:** v13.2 \
**NVIDIA driver:** v580.159.04 \
**GPU:** NVIDIA Blackwell GB203 \
**CPU:** Intel 12th Gen Core i5-12600K / 16 logical cores @ 4.90 GHz \
**DRAM:** 64 Gigabytes (~62.8 Gibibyte) at JEDEC-3200-16-20-20-38 1.35V, manufactured by Micron \
**Mass storage:** 1 Terabyte (~928.9 Gibibyte) SAMSUNG 870QVO SATA-III-600

Reporting hardware and driver versions matters for reproducibility: cuDNN (= CUDA Deep Neural Networks) and cuBLAS (= CUDA Basic Linear Algebra Subprograms) select different low-level kernels depending on GPU architecture, driver version, and CUDA toolkit version, and these kernels are not guaranteed to be bit-identical across configurations even with identical seeds and deterministic flags.

---
## 2 Determinism and its Challenges

**First of all:** strict, bit-for-bit determinism is not achievable in this notebook, and in a general GPU-accelerated setting it never fully is. What we can realistically achieve is **near-determinism**: eliminating every source of randomness that is under our control (seeds, thread counts, algorithm selection), while accepting that a residual, typically very small amount of run-to-run variation remains from sources we cannot control at the Python level (kernel scheduling, floating-point non-associativity in some GPU reduction kernels, and - where relevant - hardware/driver differences).

This notebook was developed on a local workstation where we have full control over the operating system, environment variables, and hardware. It is intended to also run on Google Colab, where we have essentially no control over any of this. Consequently, **results produced on Colab should not be expected to reproduce bit-for-bit the results produced locally**, even though both runs use the same seed and the same code.

A central practical constraint: several of the environment variables below must be set *before the Python interpreter starts*, because the libraries that read them (cuBLAS, MKL/OpenMP) cache their configuration at process/library initialization time. Setting them with `os.environ[...]` inside a running notebook only works reliably if it happens before the relevant library has been imported and initialized - and for `PYTHONHASHSEED` it does not work at all, since the hash seed is fixed by the interpreter at startup. On Colab, where we cannot control how the kernel process is launched, some of these variables cannot be set at all.

The environment variables relevant to this notebook:

| Environment Variable & Value | Reason for Change | Explanation |
|------------------------------|-------------------|-------------|
| `CUBLAS_WORKSPACE_CONFIG = ":4096:8"` | Compatibility | Required for NVIDIA CUDA ≥ 10.2 when `torch.use_deterministic_algorithms(True)` is set; otherwise cuBLAS (CUDA Basic Linear Algebra Subprograms) raises a `RuntimeError`. This configuration allocates 8 workspace buffers of 4096 Kibibyte each, which cuBLAS needs to guarantee deterministic reduction order for certain GEMM operations. |
| `OMP_NUM_THREADS = "1"` | Removing a non-deterministic reduction source | Our logistic regression pipeline uses a TF-IDF vectorizer, which is a deterministic transformation in principle. However, `scikit-learn`/`numpy` delegate the underlying linear algebra to BLAS, which parallelizes reductions (e.g. summations) across threads. Because floating-point addition is not associative, the order in which partial sums are combined affects the result at the level of the last mantissa bits - and that order depends on thread scheduling. Forcing single-threaded execution removes this source of variation. |
| `MKL_NUM_THREADS = "1"` | Same as above | Same rationale as `OMP_NUM_THREADS`, for installations where NumPy/SciPy are linked against Intel MKL rather than OpenBLAS. |
| `PYTHONHASHSEED = "12011853"` | Removing a non-deterministic reduction source | Fixes the seed of Python's string-hash randomization (enabled by default since Python 3.3 as a security hardening measure). Without this, iteration order over unordered hash-based collections such as `set` or `dict` (in older Python versions) can vary between interpreter runs, which can silently change data ordering. |

---
## 3 Setting environment variables

### 3.1 Specify env-variables in the `kernel.json` (recommended)

We set these environment variables directly in the Jupyter kernel specification (`kernel.json`), so that they are present in the process environment *before* Python - and therefore before any BLAS library - is initialized. An equivalent alternative would be a small shell wrapper script (e.g. placed under `/etc/conda/activate.d/` in the `conda` environment) that exports the variables and then execs the kernel.

This cannot be done in Google Colab, since we have no access to the kernel launch configuration there. We include our `kernel.json` below purely for reference and reproducibility reporting; it is not meant to be executed as Python code.

```json
// kernel.json - reference only, not used/usable in Google Colab
{
 "argv": [
  "/home/simon/anaconda3/envs/ml_determinism/bin/python",
  "-Xfrozen_modules=off",
  "-m",
  "ipykernel_launcher",
  "-f",
  "{connection_file}"
 ],
 "display_name": "Conda ml_determinism",
 "language": "python",
 "env": {
    "PYTHONHASHSEED": "12011853",
    "MKL_NUM_THREADS": "1",
    "OMP_NUM_THREADS": "1",
    "CUBLAS_WORKSPACE_CONFIG": ":4096:8"
    },
 "metadata": {
  "debugger": true
 },
 "kernel_protocol_version": "5.5"
}
```

### 3.2 Use `os.environ` (not safe)

Below we show, for illustration, how these variables *would* be set at runtime with `os.environ`. We do **not** actually rely on this approach in this project, ass it is not guaranteed to succeed. Furthermore, PYTHONHASHSEED cannot be set this way.

In [1]:
import os

# Illustrative only - NOT the mechanism actually used in this project.
# Setting these after the interpreter has already started is too late for
# PYTHONHASHSEED, and risky for CUBLAS_WORKSPACE_CONFIG / *_NUM_THREADS if any
# BLAS-backed library has already been imported.
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"

### 3.3 Use `bash` to run a .py-script with env-variables set (necessary for Google Colab)

We cannot change the kernel launch config in Google Colab. However, we can use `bash` to initialize a second instance of Python inside the Colab environment. In this instance we can set the env-variables during initialization. 
> **Note:** This approach will run the `Reichel_Code_deterministic.py` file from start to end without the ability to inspect code cells or run them one after another. \
> We recommend this Jupyter notebook to inspect code and evaluate it. There are many markdown cells in this notebook explaining what is happening.

If you decide to follow this approach make sure you pass the following command exactly like this to `bash`. Do not add any whitespaces or additional characters or `bash` will not recognize the command.

In [2]:
!PYTHONHASHSEED=12011853 CUBLAS_WORKSPACE_CONFIG=:4096:8 OMP_NUM_THREADS=1 MKL_NUM_THREADS=1 python Reichel_Code_deterministic.py

python: can't open file '/home/simon/Documents/Machine learning Seminar/Reichel_Code_deterministic.py': [Errno 2] No such file or directory


---
## 4 Config class

We define a `Config` dataclass that holds every important parameter as a single source of truth, which keeps the rest of the notebook free of magic numbers and makes parameters easy to control and change in one place.

> **Note:** we use `@dataclass` rather than a plain class specifically because dataclasses auto-generate `__init__`/`__repr__`/`__eq__`, and, importantly for mutable defaults like `list`, require `field(default_factory=...)` instead of a bare mutable default. A bare `list` or `dict` default on a plain class attribute would be *shared* across all instances of that class, which can lead to state-leakage bugs. We likely never instantiate `Config` more than once here, but avoiding shared mutable state is good practice regardless.

As `seed` we choose **12011853**, the birth date of Gregorio Ricci-Curbastro (12 January 1853), in honour of his foundational work on tensor calculus (the *Ricci calculus*), which underlies much of the linear-algebraic machinery used throughout modern machine learning.

In [2]:
from pathlib import Path # needed later
from dataclasses import dataclass, field

@dataclass
class Config: # single source of truth

    # data
    dataset_path: str = "/home/simon/Documents/Machine learning Seminar/Datasets/survey_vapi_messages_linked.csv"
    sb10k_train: str = "/home/simon/Documents/Machine learning Seminar/Datasets/train.tsv"
    sb10k_test: str = "/home/simon/Documents/Machine learning Seminar/Datasets/test.tsv"
    output_path: str = "/home/simon/Documents/Machine learning Seminar/annotation_data.csv"
    output_directory_logistic: str = "/home/simon/Documents/Machine learning Seminar/Models/Logistic Regression Models Deterministic"
    output_directory_encoder: str = "/home/simon/Documents/Machine learning Seminar/Models/BERT Transformer Models Deterministic"
    output_directory_foundation: str = "/home/simon/Documents/Machine learning Seminar/Models/Foundation Models Deterministic"
    dataset_columns: list[str] = field(default_factory = lambda: ["interviewId",
                                                                  "messageId",
                                                                  "role",
                                                                  "message"])
    seed: int = 12011853
    sample_size: int = 100  # gold-standard validation sample size

    # inference dataset
    min_words: int = 10  # minimum number of words per message

    # logistic regression
    tfidf_params: dict = field(default_factory = lambda: {"ngram_range": (1, 1), # unigrams + bigrams
                                                          "max_features": 20000,
                                                         "sublinear_tf": True}) # apply log-scaling

    clf_params: dict = field(default_factory = lambda: {"C": 1.0, # regularization strength
                                                       "max_iter": 1000,
                                                       "solver": "lbfgs"})

    #encoder transformer
    name: str = "BERT-Base-German"
    model_checkpoint: str = "google-bert/bert-base-german-cased"
    num_labels: int = 3 # number of classes
    max_length: int = 128 # max number of tokens
    learning_rate: float = 2e-5
    batch_size: int = 16
    num_epochs: int = 3
    weight_decay: float = 0.01
    patience: int = 2 #training will stop early if fulfilled
    dataloader_num_workers: int = 0 # avoid non-determinism through worker reseeding
    use_bf16: bool = True # set to True for mixed precision with BF16
    use_tf32: bool = True # set to true for faster 32-bit matmuls

    #instruct-based foundational model with LoRA
    model_checkpoint_foundation: str = "Qwen/Qwen2.5-0.5B-Instruct" # small model to save compute
    max_length_foundation: int = 512
    learning_rate_foundation: float = 1e-4 #higher than BERT because of LoRA
    batch_size_foundation: int = 8
    num_epochs_foundation: int = 3
    weight_decay_foundation: float = 0.01
    lora_r: int = 16
    lora_alpha: int = 32
    lora_dropout: float = 0.05
    label_set_foundation: tuple = ("negativ", "neutral", "positiv") # three classes
    do_sample: bool = False # set False for greedy decoding
    use_bf16_foundation: bool = True # set to True for mixed precision with BF16
    use_tf32_foundation: bool = True # set to true for faster 32-bit matmuls
    dataloader_num_workers_foundation: int = 0

    
    
    # set near-determinism
    deterministic: bool = True

    # technical parameters
    device: str = "cuda"


cfg = Config()


In [4]:
import torch

# specify torch device
device = torch.device(cfg.device)

# sanity-check the installed versions against the ones this notebook was developed with
print(f"CUDA version: {torch.version.cuda} (developed against: 13.2)")
print(f"torch version: {torch.__version__} (developed against: 2.13.0)")

CUDA version: 13.2 (developed against: 13.2)
torch version: 2.13.0+cu132 (developed against: 2.13.0)


We can already specify our CUDA device as a `torch` device for later use, and take the opportunity to sanity-check the installed CUDA/PyTorch versions against what this notebook was developed against.

---
## 5 Define `set_determinism()` function

We define `set_determinism()`, a function that configures every seed and every determinism-relevant flag we have programmatic control over. Even with this function applied, a small amount of non-determinism remains: OS-level thread/kernel scheduling and certain GPU reduction kernels are inherently non-deterministic and are out of reach from user-space Python.

Linux Fedora would in principle let us swap in a fully deterministic kernel, but we intentionally do not go that far, for two reasons:
- Modifying the kernel is delicate and can break the system; we would only recommend this to advanced users who specifically need it.
- Windows, macOS, and many Linux distributions do not expose this level of control at all, so this step would not be reproducible by most readers.

\
`set_determinism()` sets the following, when `cfg.deterministic is True`:

- **`torch.manual_seed(seed)`** seeds PyTorch's CPU (and, as a side effect, default CUDA) random number generator. Not every PyTorch operation consumes randomness from this generator, which is why several more seeds are needed below.
  
- **`torch.cuda.manual_seed_all(seed)`** seeds the CUDA RNG on *all* visible GPUs (the `_all` suffix matters in multi-GPU settings; the non-suffixed variant only seeds the current device).

- **`np.random.seed(seed)`** seeds NumPy's legacy global RNG, which many libraries (including parts of scikit-learn) depend on internally.

- **`random.seed(seed)`** seeds Python's standard-library `random` module, used implicitly by some libraries without this being obvious from their public API.

- **`torch.use_deterministic_algorithms(True, warn_only=True)`** instructs PyTorch to prefer deterministic kernel implementations wherever they exist. Some operations (e.g. certain scatter/gather patterns, some CUDA reduction kernels) have no deterministic implementation at all. With `warn_only=True`, PyTorch falls back to the non-deterministic implementation and emits a warning instead of raising `RuntimeError`; with `warn_only=False` it would raise instead.

- **`torch.utils.deterministic.fill_uninitialized_memory = True`** is a *property assignment*, not a function call - this is a small but easy mistake to make (an earlier version of this code incorrectly attempted `fill_uninitialized_memory(True)`, treating it like `torch.use_deterministic_algorithms()`). When set, newly allocated tensors that PyTorch would otherwise leave with arbitrary uninitialized memory (e.g. from `torch.empty`) are instead deterministically zero-filled. Left at its default, the *contents* of such tensors are undefined and can vary between runs - not because anything is being computed non-deterministically, but because the memory simply was never written.

- **`torch.backends.cudnn.deterministic = True`** forces cuDNN to use only its deterministic convolution/pooling algorithm implementations. (An earlier version of this notebook stated this flag as `= False` while describing the deterministic behaviour - that was a copy-paste error in the markdown, not in the code; the code cell below has always used `= True` in the deterministic branch.)

- **`torch.backends.cudnn.benchmark = False`** disables cuDNN's autotuning heuristic, which would otherwise time several candidate algorithms on the first forward pass and cache whichever is fastest - a choice that can itself depend on non-deterministic timing and on input shapes seen so far. Disabling it makes cuDNN pick algorithms deterministically based on the operation's static parameters instead.

- **`generator = torch.Generator().manual_seed(seed)`** creates a seeded generator object intended for PyTorch `DataLoader`. Setting the global seeds above is not sufficient for `DataLoader` workers: when `num_workers > 0`, each worker process is forked and reseeded independently, so **this `generator` object must be passed explicitly** as `DataLoader(..., generator=generator, worker_init_fn=seed_worker)` for worker-level reproducibility - the seeds set above do not propagate into worker processes on their own. We return `generator` from this function precisely so it can be threaded through to any `DataLoader` used later (Tasks 2 and 3).

When `cfg.deterministic is False`, the mirrored branch draws a fresh pseudo-random seed via `random.SystemRandom()` and disables all of the above, additionally re-enabling `cudnn.benchmark` so that cuDNN is free to autotune for speed.

In [24]:
import numpy as np
import random

def set_determinism(cfg):

    if cfg.deterministic is True:

        # take seed from Config
        seed = cfg.seed

        # torch seed fixed
        torch.manual_seed(seed)

        # NVIDIA CUDA seed fixed (all visible devices)
        torch.cuda.manual_seed_all(seed)

        # numpy seed fixed
        # numpy is a dependency in many modules
        np.random.seed(seed)

        # random seed fixed
        # some functions might use this without specifically saying so
        random.seed(seed)

        # prefer deterministic algorithms where they exist
        # explicitly fore determinism by warn_only = False
        torch.use_deterministic_algorithms(True,
                                           warn_only = False)

        # force newly allocated tensors to be zero-filled rather than left
        # with arbitrary uninitialized memory contents
        torch.utils.deterministic.fill_uninitialized_memory = True

        # restrict NVIDIA cuDNN to deterministic algorithm implementations
        torch.backends.cudnn.deterministic = True

        # disable cuDNN's autotuning benchmark so algorithm choice is static,
        # not the (non-deterministic) fastest-of-several-timed-runs
        torch.backends.cudnn.benchmark = False

        # seeded generator for DataLoader workers - must be passed explicitly
        # to DataLoader(..., generator=generator, worker_init_fn=seed_worker)
        generator = torch.Generator().manual_seed(seed)

    else:
        # choose a pseudo-random 32-bit integer
        seed = random.SystemRandom().randint(0, 2**31 - 1)

        # torch seed
        torch.manual_seed(seed)

        # NVIDIA CUDA seed
        torch.cuda.manual_seed_all(seed)

        # numpy seed
        np.random.seed(seed)

        # random seed
        # some functions might use this without specifically saying so
        random.seed(seed)

        # allow torch to fall back to non-deterministic (typically faster) algorithms
        torch.use_deterministic_algorithms(False)

        # tensors may contain arbitrary uninitialized (non-deterministic) values
        torch.utils.deterministic.fill_uninitialized_memory = False

        # allow cuDNN to select non-deterministic algorithms
        torch.backends.cudnn.deterministic = False

        # let cuDNN autotune and cache the fastest algorithm per input shape
        torch.backends.cudnn.benchmark = True

        # generator still seeded, just not reproducible across runs
        generator = torch.Generator().manual_seed(seed)

    print(f"Operations are near-deterministic: {cfg.deterministic}")
    print(f"Seed: {seed}")

    return seed, generator

---
## 6 Training dataset pre-processing

For training we use the SB10k dataset for three-class sentiment analysis by Cieliebak et al. (2017). \
The dataset already has train and test splits.

In [6]:
import pandas as pd

# read datasets
train_split = pd.read_table(cfg.sb10k_train,
                            usecols = [1, 2],
                            names = ["rating", "text"]) # specify column headers

test_split = pd.read_table(cfg.sb10k_test,
                            usecols = [1, 2],
                            names = ["rating", "text"]) # specify column headers


print(f"Rows train_split: {len(train_split)}")
print(f"Rows test_split: {len(test_split)}")
print(f"Split ratio: {len(test_split) / len(train_split)}")
train_split.head()

Rows train_split: 5233
Rows test_split: 1496
Split ratio: 0.285878081406459


,rating,text
0,positive,RT @TheKedosZone : So ein Hearthstone - Key vo...
1,neutral,"Tainted Talents ( Ateliertagebuch. ) "" Wir sin..."
2,neutral,Aber wenigstens kommt #Supernatural heute mal ...
3,neutral,DARLEHEN - Angebot für Schufa - freie Darlehen...
4,neutral,ANRUF ERWÜNSCHT : Hardcore Teeny Vicky Carrera...


Per convention, our split names are:
- `X_train`: strings for training
- `X_test`: strings for validation
- `y_train`: labels for training
- `y_test`: labels for validation

In [7]:
# transform splits
X_train = train_split["text"]
X_test = test_split["text"]

y_train = train_split["rating"]
y_test = test_split["rating"]

# look at distributions
print(f"Train split class distributions: {y_train.value_counts()}")
print(f"Test split class distributions: {y_test.value_counts()}")

Train split class distributions: rating
neutral     3246
positive    1193
negative     794
Name: count, dtype: int64
Test split class distributions: rating
neutral     930
positive    354
negative    212
Name: count, dtype: int64


---
## 7 Inference dataset pre-processing

In [8]:
# read data
dataset_inference = pd.read_csv(cfg.dataset_path)

# verify
dataset_inference.info()

<class 'pandas.DataFrame'>
RangeIndex: 8627 entries, 0 to 8626
Columns: 151 entries, StartDate to analysis_summary
dtypes: bool(1), float64(69), int64(3), object(2), str(76)
memory usage: 30.7+ MB


/tmp/ipykernel_87658/1830462024.py:2: DtypeWarning: Columns (0: Q91_2_TEXT, 1: eval_compassionate, 2: q42_2, 3: q20_10_TEXT, 4: q22_10_TEXT, 5: q75_10_TEXT, 6: PROLIFIC_PID, 7: stereoRecordingUrl) have mixed types. Specify dtype option on import or set low_memory=False.
  dataset_inference = pd.read_csv(cfg.dataset_path)


In [9]:
# drop unnecessary columns
dataset_reduced = dataset_inference[cfg.dataset_columns]

# verify
dataset_reduced.head()

,interviewId,messageId,role,message
0,afaf2cf9-66c3-44e6-b183-44f704e8278b,afaf2cf9-66c3-44e6-b183-44f704e8278b_msg_001,system,# Rolle\n\nSie sind ein kompetenter Interviewe...
1,afaf2cf9-66c3-44e6-b183-44f704e8278b,afaf2cf9-66c3-44e6-b183-44f704e8278b_msg_002,bot,"Hallo, Ich freue mich, heute mit Ihnen spreche..."
2,afaf2cf9-66c3-44e6-b183-44f704e8278b,afaf2cf9-66c3-44e6-b183-44f704e8278b_msg_003,user,"Er Ja, wir können starten."
3,afaf2cf9-66c3-44e6-b183-44f704e8278b,afaf2cf9-66c3-44e6-b183-44f704e8278b_msg_004,bot,"Herzlich willkommen und vielen Dank, dass Sie ..."
4,afaf2cf9-66c3-44e6-b183-44f704e8278b,afaf2cf9-66c3-44e6-b183-44f704e8278b_msg_005,user,Ich möchte direkt loslegen.


We are only interested in user messages, since these already constitute natural-language entities on their own (as opposed to system prompts or structured assistant replies). \
We additionally drop messages with fewer than `cfg.min_words` words, since very short replies rarely carry enough content to be judged reliably for sentiment - note that this is a **word-count** threshold (`str.split().str.len()` counts whitespace-separated tokens), not a character-length threshold.

We then draw a pseudo-random sample for manual annotation, which will later serve as real-world validation data across all three architectures.

In [10]:
# select only user messages
dataset_user = dataset_reduced[dataset_reduced["role"] == "user"]

# keep only messages with at least cfg.min_words words
dataset_user = dataset_user[dataset_user["message"].str.split().str.len() >= cfg.min_words]

# check dataset size
print(f"Remaining messages: {len(dataset_user)}")

# pull pseudo-random sample
dataset_annotations = dataset_user.sample(n = cfg.sample_size,
                                          random_state = cfg.seed
                                         )

# check annotation dataset size
print(f"Annotation dataset size: {len(dataset_annotations)}")

Remaining messages: 2376
Annotation dataset size: 100


In [11]:
# export to .csv
dataset_annotations.to_csv(cfg.output_path,
                           encoding = "UTF-8",
                           index = False,
                           sep = "|"  # unlikely to appear in the data
                          )

---
## 8 Model training

We train on three different architectures. First, we are training from scratch on a logistic regression classifier. Afterwards we are applying transfer learning on a BERT-like transformer. Finally, we apply transfer learning on a small Large Language Model. \
We purposefully do not conduct k-fold cross validation to ensure all models within each architecture are trained on the exact same data, eliminating a source of noise. \
\
To evaluate differences in training and inference due to non-deterministic or near-deterministic environments, we train ten models for each architecture and inference over the dataset 200 times for each model. \
Therefore, we will inference the dataset a total of 12,000 times (60 models with 200 runs each), giving us a broad basis for statistical tests.

Before running any of the three tasks, we call `set_determinism()`. Most of its effects are irrelevant to logistic regression (there is no cuDNN, no CUDA tensor allocation, no `DataLoader`). However, they become relevant starting with Task 2, which uses a GPU-resident transformer encoder.

In [25]:
# set calculations to be near-deterministic
seed, generator = set_determinism(cfg)

Operations are near-deterministic: True
Seed: 12011853


### 8.1 Logistic regression

The determinism-relevant parameters here:

- The `'lbfgs'` solver used here is a deterministic quasi-Newton optimizer that starts from a fixed (zero) coefficient initialization and follows a deterministic sequence of updates for a given input.
- We use a `TfidfVectorizer`, which is a deterministic transformation of text into vectors given fixed input. However, the underlying sparse-matrix and normalization arithmetic in `scikit-learn`/`numpy` is dispatched to BLAS, which parallelizes reductions across threads. Because floating-point addition is not associative, the reduction order and hence the least-significant bits of the result can vary with thread scheduling. We already forced `OMP_NUM_THREADS=1` and `MKL_NUM_THREADS=1` at the very start of this notebook specifically to remove this source of variation.

> **Note:** single-threaded BLAS execution is significantly slower than the multi-threaded default.

\
We use `nltk` to remove German stopwords:

In [10]:
import json
import joblib
import sklearn
import platform
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, 
    precision_recall_fscore_support,
    classification_report, 
    confusion_matrix
)
import nltk
from nltk.corpus import stopwords
from time import perf_counter
from datetime import datetime


# download German stopwords
nltk.download('stopwords')

german_stopwords = stopwords.words('german')

[nltk_data] Downloading package stopwords to /home/simon/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Because we want to train the model multiple times, we define a set of functions. \
We define a pipeline, an evaluation function for the most important metrics and a `fit_and_save` function which will fit the model and then save it to our storage.

In [14]:
# define logistic regression pipeline
def build_pipeline(tfidf_params = None, 
                   clf_params = None):
    
    tfidf_params = tfidf_params or {}
    clf_params = clf_params or {}
    
    return Pipeline([
        ("tfidf", TfidfVectorizer(**tfidf_params)),
        ("clf", LogisticRegression(**clf_params)),
    ])

In [15]:
# define evaluation function
def evaluate(pipeline, 
             X_test, 
             y_test):
    
    y_pred = pipeline.predict(X_test)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_test, y_pred, average = "weighted", zero_division = 0
    )
    
    return {
        "accuracy": accuracy_score(y_test, y_pred),
        "precision_weighted": precision,
        "recall_weighted": recall,
        "f1_weighted": f1,
        "classification_report": classification_report(
            y_test, y_pred, output_dict = True, zero_division = 0
        ),
        "confusion_matrix": confusion_matrix(y_test, y_pred).tolist(),
    }

In [16]:
# define fitting model and saving to storage
def fit_and_save(name, 
                 tfidf_params, 
                 clf_params, 
                 X_train, 
                 y_train, 
                 X_test, 
                 y_test):
    
    pipeline = build_pipeline(tfidf_params, 
                              clf_params)

    start = perf_counter()
    pipeline.fit(X_train, y_train)
    fit_seconds = perf_counter() - start

    metrics = evaluate(pipeline, 
                       X_test, 
                       y_test)

    # save model
    model_path = Path(cfg.output_directory_logistic) / f"{name}.joblib"
    joblib.dump(pipeline, model_path)

    # save metadata
    metadata = {
        "name": name,
        "timestamp": datetime.now().isoformat(),
        "tfidf_params": tfidf_params,
        "clf_params": clf_params,
        "fit_seconds": fit_seconds,
        "n_train": len(X_train),
        "n_test": len(X_test),
        "metrics": metrics,
        "model_path": str(model_path),

        # versions
        "scikit learn version": sklearn.__version__, 
        "Python version": platform.python_version()
    }
    
    metadata_path = Path(cfg.output_directory_logistic) / f"{name}_metadata.json"
    with open(metadata_path, "w") as f:
        json.dump(metadata, f, indent = 2)

    print(f"[{name}] acc = {metrics['accuracy']:.4f}  f1 = {metrics['f1_weighted']:.4f} fitted in {fit_seconds:.2f} seconds and was saved to {model_path}")

    return pipeline, metadata

We can now run training.

In [17]:
results_logistic = {}

# fit ten models
for x in range(10):

    #drop pipeline with _
    _, meta = fit_and_save("logistic_regression_deterministic_" + str(x),
                           cfg.tfidf_params,
                           cfg.clf_params,
                           X_train,
                           y_train,
                           X_test,
                           y_test
                          )
    results_logistic[x] = meta

with open(Path(cfg.output_directory_logistic) / "summary_deterministic.json", "w") as f:
    json.dump(
        {k: {"accuracy": v["metrics"]["accuracy"],
             "f1_weighted": v["metrics"]["f1_weighted"]}
         for k, v in results_logistic.items()},
        f, 
        indent = 2
    )

[logistic_regression_deterministic_0] acc = 0.6898  f1 = 0.6466 fitted in 0.21 seconds and was saved to /home/simon/Documents/Machine learning Seminar/Models/Logistic Regression Models Deterministic/logistic_regression_deterministic_0.joblib
[logistic_regression_deterministic_1] acc = 0.6898  f1 = 0.6466 fitted in 0.21 seconds and was saved to /home/simon/Documents/Machine learning Seminar/Models/Logistic Regression Models Deterministic/logistic_regression_deterministic_1.joblib
[logistic_regression_deterministic_2] acc = 0.6898  f1 = 0.6466 fitted in 0.20 seconds and was saved to /home/simon/Documents/Machine learning Seminar/Models/Logistic Regression Models Deterministic/logistic_regression_deterministic_2.joblib
[logistic_regression_deterministic_3] acc = 0.6898  f1 = 0.6466 fitted in 0.20 seconds and was saved to /home/simon/Documents/Machine learning Seminar/Models/Logistic Regression Models Deterministic/logistic_regression_deterministic_3.joblib
[logistic_regression_determinist

### 8.2 Encoder Transformer

In [22]:
import numpy as np
import accelerate
from datasets import Dataset
from torch.utils.data import DataLoader
import accelerate.utils
import transformers as tf

We define a `compute_metrics` function which does essentially the same as the `evaluate` function we defined for logistic regression.

In [19]:
def compute_metrics(eval_pred):
    
    logits, labels = eval_pred
    preds = np.argmax(logits, axis = -1)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average = "weighted", zero_division = 0
    )
    
    return {
        "accuracy": accuracy_score(labels, preds),
        "precision_weighted": precision,
        "recall_weighted": recall,
        "f1_weighted": f1,
    }

Define the training loop and checkpoint logging:

In [34]:
def fit_and_save_transformer(cfg,
                             name,
                             X_train, 
                             y_train,
                             X_test, 
                             y_test, 
                             test_texts = None, 
                             test_labels = None):
    
    tf.set_seed(cfg.seed)  # seed for reproducibility

    # set output directory
    run_dir = Path(cfg.output_directory_encoder) / name
    # verify it exists
    run_dir.mkdir(parents = True, exist_ok = True)
    
    tokenizer = tf.AutoTokenizer.from_pretrained(cfg.model_checkpoint)
    model = tf.AutoModelForSequenceClassification.from_pretrained(
        cfg.model_checkpoint, 
        num_labels = cfg.num_labels # number of classes
    )

    def tokenize(batch):
        
        return tokenizer(batch["text"], 
                         truncation = True, 
                         padding = "max_length",
                         max_length = cfg.max_length)

    #convert splits to list for trainer
    train_texts = X_train.reset_index(drop = True).tolist()
    train_labels = y_train.reset_index(drop = True).tolist()
    test_texts = X_test.reset_index(drop = True).tolist()
    test_labels = y_test.reset_index(drop = True).tolist()
    
    train_ds = Dataset.from_dict({"text": train_texts, "label": train_labels}).map(
        tokenize, batched = True
    )
    eval_ds = Dataset.from_dict({"text": test_texts, "label": test_labels}).map(
        tokenize, batched = True
    )

    training_args = tf.TrainingArguments(
        output_dir = str(run_dir / "checkpoints"),
        learning_rate = cfg.learning_rate,
        per_device_train_batch_size = cfg.batch_size,
        per_device_eval_batch_size = cfg.batch_size,
        num_train_epochs = cfg.num_epochs,
        weight_decay = cfg.weight_decay,
        eval_strategy = "epoch",
        save_strategy = "epoch",
        save_total_limit = 2, # keep only last 2 checkpoints to save memory
        load_best_model_at_end = True,
        metric_for_best_model = "f1_weighted",
        seed = cfg.seed,
        data_seed = cfg.seed,
        report_to = "none",
        TENSORBOARD_LOGGING_DIR = str(run_dir / "logs"),
        bf16 = cfg.use_bf16, # mixed precision on Blackwell
        tf32 = cfg.use_tf32, # speeds up fp32 matmuls on Blackwell
        dataloader_pin_memory = True
    )

    # use trainer
    trainer = tf.Trainer(
            model = model,
            args = training_args,
            train_dataset = train_ds,
            eval_dataset = eval_ds,
            compute_metrics = compute_metrics,
            callbacks = [tf.EarlyStoppingCallback(early_stopping_patience = cfg.patience)]
        )

    start = perf_counter()
    trainer.train()
    fit_seconds = perf_counter() - start

    eval_metrics = trainer.evaluate()

    # Save the final best model
    final_dir = run_dir / "final_model"
    trainer.save_model(str(final_dir))
    tokenizer.save_pretrained(str(final_dir))

    # Use annotated gold-standard
    test_metrics = None
    if test_texts is not None:
        test_ds = Dataset.from_dict({"text": test_texts, "label": test_labels}).map(
            tokenize, batched = True
        )
        
        preds_output = trainer.predict(test_ds)
        preds = np.argmax(preds_output.predictions, axis = -1)
        precision, recall, f1, _ = precision_recall_fscore_support(
            test_labels, 
            preds, 
            average = "weighted", 
            zero_division = 0
        )

        # will only be created if test_teesxts exist
        test_metrics = {
            "accuracy": accuracy_score(test_labels, preds),
            "precision_weighted": precision,
            "recall_weighted": recall,
            "f1_weighted": f1,
            "confusion_matrix": confusion_matrix(test_labels, preds).tolist(),
        }

    
    metadata = {
        "name": name,
        "timestamp": datetime.now().isoformat(),
        "model_checkpoint": cfg.model_checkpoint,
        "hyperparameters": {
            "learning_rate": cfg.learning_rate,
            "batch_size": cfg.batch_size,
            "num_epochs": cfg.num_epochs,
            "weight_decay": cfg.weight_decay,
            "max_length": cfg.max_length,
        },
        "fit_seconds": fit_seconds,
        "eval_metrics": eval_metrics,
        "test_metrics": test_metrics,
        "final_model_path": str(final_dir),
        "CUDA_device_memory_used": torch.cuda.memory_reserved(0), # get info about needed GPU memory
        "torch_version": torch.__version__,
    }

    # save metadata as json
    with open(run_dir / "metadata.json", "w") as f:
        json.dump(metadata, 
                  f, 
                  indent = 2)

    print(f"[{name}] eval_f1={eval_metrics.get('eval_f1_weighted', 'NA')} fitted in {fit_seconds:.2f} seconds and was saved to {final_model_path}")

    return trainer, metadata

Before we can run inference, we need to change our labels from strings to integers. Otherwise `torch` will not be able to convert them into a tensor.

In [8]:
y_train_int = y_train.replace(
    ["negative",
    "neutral",
    "positive"],
    [0,
    1,
    2]
)

y_test_int = y_test.replace(
    ["negative",
    "neutral",
    "positive"],
    [0,
    1,
    2]
)

In [35]:
results_encoder = {}

# fit ten models
for x in range(10):
    _, meta = fit_and_save_transformer(
        cfg,
        "BERT-Base-German_" + str(x),
        X_train, 
        y_train_int, # labels converted to integers
        X_test, 
        y_test_int, # labels converted to integers
        test_texts = None, 
        test_labels = None      
    )

    results_encoder[x] = meta

    #free up GPU memory between runs
    del _
    torch.cuda.empty_cache()

summary = {
    name: {
        "eval_f1": meta["eval_metrics"].get("eval_f1_weighted"),
        "test_f1": (meta["test_metrics"] or {}).get("f1_weighted"),
        "fit_seconds": meta["fit_seconds"],
    }
    
    for name, meta in results_encoder.items()
}

with open(Path(cfg.output_directory_encoder), "w") as f:
    json.dump(summary,
              f,
              indent = 2)

#free up GPU memory at the end
del _
torch.cuda.empty_cache()

Loading weights: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 199/199 [00:00<00:00, 15860.65it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: google-bert/bert-base-german-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different t

Epoch,Training Loss,Validation Loss,Accuracy,Precision Weighted,Recall Weighted,F1 Weighted
1,No log,0.605425,0.745321,0.744010,0.745321,0.727932
2,0.617141,0.618394,0.763369,0.756707,0.763369,0.757293
3,0.617141,0.711569,0.764037,0.757364,0.764037,0.759822


Writing model shards: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  3.52it/s]


Training Loss,Validation Loss,Epoch,Accuracy,Precision Weighted,Recall Weighted,F1 Weighted
0.617141,0.711569,3,0.764037,0.757364,0.764037,0.759822


Writing model shards: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  3.41it/s]


...

Writing model shards: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  3.65it/s]


Training Loss,Validation Loss,Epoch,Accuracy,Precision Weighted,Recall Weighted,F1 Weighted
0.617141,0.711569,3,0.764037,0.757364,0.764037,0.759822


Writing model shards: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  3.51it/s]

[BERT-Base-German_9] eval_f1=0.7598219443741256  saved -> /home/simon/Documents/Machine learning Seminar/Models/BERT Transformer Models Deterministic/BERT-Base-German_9/final_model


KeyError: 'test_metrics'

### 8.3 Large Language Model

Finally, we want to train some foundational models. We use a pre-trained checkpoint and use Low Rank Adaptation (LoRA) for conversation style training. \
First of all, we define a system prompt for the model:

In [11]:
SYSTEM_PROMPT = ("Du bist ein hilfreicher Assistent, der die Stimmung (Sentiment) eines deutschen Textes klassifiziert. Antworte ausschließlich mit einem Wort: negativ, neutral, oder positiv.")

In [12]:
from peft import LoraConfig, get_peft_model, TaskType

/home/simon/anaconda3/envs/ml_determinism/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


We build `build_chat_template`. Note that we set `tokenize = False` on `apply_chat_format`. If it is set to `True` it might overflow.

In [13]:
def build_chat_example(tokenizer, 
                       text, 
                       label, 
                       max_length):
    
    """Formats one example using the model chat template, keeping the
    conversational structure intact, and masks the prompt tokens in the labels
    so loss is only computed on the assistant answer."""

    # force label to be a string
    label_str = str(label)
    
    messages_prompt = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": text},
    ]

    # define prompt format
    # contains system and user
    prompt_str = tokenizer.apply_chat_template(
        messages_prompt, 
        tokenize = False, # set to False to force string templating only 
        add_generation_prompt = True
    )

    # define full messages
    # contains system, user and assistant
    full_messages = messages_prompt + [{"role": "assistant", "content": label_str}]
    full_str = tokenizer.apply_chat_template(
        full_messages, 
        tokenize = False, # set to False to force string templating only 
        add_generation_prompt = False
    )

    # tokenize
    prompt_ids = tokenizer(prompt_str,
                           add_special_tokens = False)["input_ids"]
    full_ids = tokenizer(full_str,
                         add_special_tokens = False)["input_ids"]

    full_ids = full_ids[:max_length]
    labels = list(full_ids)
    prompt_len = min(len(prompt_ids), len(full_ids))
    
    for i in range(prompt_len):
        labels[i] = -100  # mask prompt tokens from the loss

    
    return {

        # force integers to be sure
        "input_ids": [int(i) for i in full_ids],
        "labels": [int(l) for l in labels],
    }

We can now use `build_chat_example` to build a dataset.

In [14]:
def build_dataset(tokenizer, 
                  texts, 
                  labels, 
                  max_length):
    
    #do a list comprehension for examples
    examples = [build_chat_example(tokenizer, t, l, max_length) for t, l in zip(texts, labels)]
    
    return Dataset.from_list(examples)

In [15]:
class PaddingCollator:
    
    """Pads input_ids and labels to the longest sequence in the batch (-100 for
    padded label positions so they don't contribute to loss)."""
    
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer
        self.pad_id = tokenizer.pad_token_id or tokenizer.eos_token_id

    def __call__(self, features):
        max_len = max(len(f["input_ids"]) for f in features)

        # define empty lists to fill later
        input_ids, attention_mask, labels = [], [], []
        
        for f in features:
            pad_len = max_len - len(f["input_ids"])
            input_ids.append(f["input_ids"] + [self.pad_id] * pad_len)
            attention_mask.append([1] * len(f["input_ids"]) + [0] * pad_len)
            labels.append(f["labels"] + [-100] * pad_len)
        
        return {
            
            # return all as signed 64-bit integers
            "input_ids": torch.tensor(input_ids, dtype = torch.long),
            "attention_mask": torch.tensor(attention_mask, dtype = torch.long),
            "labels": torch.tensor(labels, dtype = torch.long),
        }

We define `generate_predictions`. Note that we set `tokenizer.padding_side = "left"` for predictions but set `tokenizer.padding_side = "right"` for training. \
It is necessary to do so in decoder-only models. `model.generate()` requires right padding, because decoder-only models will only attend at tokens before the current token. No PAD-tokens should be there or the model will be confused.

In [16]:
@torch.no_grad()
def generate_predictions(model, 
                         tokenizer, 
                         texts, 
                         label_set, 
                         max_new_tokens = 5, 
                         batch_size = 16):
    
    #no training
    model.eval()

    # needed for correct batched generation
    original_padding_side = tokenizer.padding_side
    tokenizer.padding_side = "left"
    
    preds = []
    
    for i in range(0, len(texts), batch_size):
        
        batch_texts = texts[i:i + batch_size]

        # list comprehension for prompts
        prompts = [
            tokenizer.apply_chat_template(
                [{"role": "system", "content": SYSTEM_PROMPT},
                 {"role": "user", "content": t}],
                tokenize = False, 
                add_generation_prompt = True,
            )
            for t in batch_texts
        ]

        # create tensors and send them to device
        inputs = tokenizer(prompts, 
                           return_tensors = "pt", 
                           padding = True).to(device) #device was defined in the beginning
        
        # create outputs
        out = model.generate(
            **inputs, max_new_tokens = max_new_tokens, do_sample = cfg.do_sample
        )
        decoded = tokenizer.batch_decode(out[:, inputs["input_ids"].shape[1]:],
                                          skip_special_tokens = True)
        
        for d in decoded:
            d_clean = d.strip().lower()
            match = next((lab for lab in label_set if lab in d_clean), label_set[0])
            
            # add results to preds
            preds.append(match)

    # restore original padding
    tokenizer.padding_side = original_padding_side
    
    return preds

We define a similar evaluation function as in logistic regression and encoder transformer.

In [32]:
def evaluate_generation(model, 
                        tokenizer, 
                        texts, 
                        true_labels, 
                        label_set):

    # use generate_predictions from above
    preds = generate_predictions(model, 
                                 tokenizer, 
                                 texts, 
                                 label_set)
    
    precision, recall, f1, _ = precision_recall_fscore_support(
        true_labels, 
        preds, 
        average = "weighted", 
        zero_division = 0, 
        labels = list(label_set)
    )
    
    
    return {
        "accuracy": accuracy_score(true_labels, preds),
        "precision_weighted": precision,
        "recall_weighted": recall,
        "f1_weighted": f1,
        "confusion_matrix": confusion_matrix(true_labels, 
                                             preds, 
                                             labels = list(label_set)).tolist(),
        "raw_predictions_sample": preds[:20],
    }

In [33]:
# specifically designed for Qwen 2
def fit_and_save_qwen(cfg,
                      name,
                      train_texts, 
                      train_labels, 
                      eval_texts, 
                      eval_labels,
                      test_texts = None, 
                      test_labels = None):

    # set seeds to be sure
    tf.set_seed(cfg.seed)
    generator = torch.Generator().manual_seed(cfg.seed)

    # define directory to use
    run_dir = Path(cfg.output_directory_foundation) / name
    
    #make sure the directory exists
    run_dir.mkdir(parents = True, exist_ok = True)

    tokenizer = tf.AutoTokenizer.from_pretrained(cfg.model_checkpoint_foundation)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    # needed for decoder-only models
    tokenizer.padding_side = "right"

    base_model = tf.AutoModelForCausalLM.from_pretrained(
        cfg.model_checkpoint_foundation, 
        dtype = torch.bfloat16 #half precision
    )

    # define LoRA config
    lora_config = LoraConfig(
        task_type = TaskType.CAUSAL_LM,
        r = cfg.lora_r,
        lora_alpha = cfg.lora_alpha,
        lora_dropout = cfg.lora_dropout,
        
        # attention modules for Qwen 2
        target_modules = ["q_proj", 
                          "k_proj", 
                          "v_proj", 
                          "o_proj"]
    )
    
    model = get_peft_model(base_model, 
                           lora_config)
    
    # build train dataset with function from above
    train_ds = build_dataset(tokenizer, 
                             train_texts, 
                             train_labels, 
                             cfg.max_length_foundation)

    # build test dataset with function from above
    eval_ds = build_dataset(tokenizer, 
                            eval_texts, 
                            eval_labels, 
                            cfg.max_length_foundation)
    
    collator = PaddingCollator(tokenizer)

    training_args = tf.TrainingArguments(
        output_dir = str(run_dir / "checkpoints"),
        learning_rate = cfg.learning_rate_foundation,
        per_device_train_batch_size = cfg.batch_size_foundation,
        per_device_eval_batch_size = cfg.batch_size_foundation,
        num_train_epochs = cfg.num_epochs_foundation,
        weight_decay = cfg.weight_decay_foundation,
        eval_strategy = "epoch",
        save_strategy = "epoch",
        save_total_limit = 2,
        seed = cfg.seed,
        data_seed = cfg.seed,
        dataloader_num_workers = cfg.dataloader_num_workers_foundation,
        report_to = "none",
        TENSORBOARD_LOGGING_DIR = str(run_dir / "logs"),
        bf16 = cfg.use_bf16_foundation,
        tf32 = cfg.use_tf32_foundation,
        dataloader_pin_memory = True,
    )

    trainer = tf.Trainer(
        model = model,
        args = training_args,
        train_dataset = train_ds,
        eval_dataset = eval_ds,
        data_collator = collator
    )

    start = perf_counter()
    trainer.train()
    fit_seconds = perf_counter() - start

    # save only LoRA adapter
    final_dir = run_dir / "final_adapter"
    model.save_pretrained(str(final_dir))
    tokenizer.save_pretrained(str(final_dir))

    # use function from above
    eval_metrics = evaluate_generation(model, 
                                       tokenizer, 
                                       eval_texts, 
                                       eval_labels, 
                                       cfg.label_set_foundation)

    # use for gold-standard annotations
    test_metrics = None
    if test_texts is not None:
        test_metrics = evaluate_generation(model, 
                                           tokenizer, 
                                           test_texts, 
                                           test_labels, 
                                           cfg.label_set)

    # create metadata dict
    metadata = {
            "name": name,
            "timestamp": datetime.now().isoformat(),
            "model_checkpoint": cfg.model_checkpoint_foundation,
            "random_state": cfg.seed,
            "lora_config": {"r": cfg.lora_r, 
                            "alpha": cfg.lora_alpha, 
                            "dropout": cfg.lora_dropout},
            "hyperparameters": {
                "learning_rate": cfg.learning_rate_foundation,
                "batch_size": cfg.batch_size_foundation,
                "num_epochs": cfg.num_epochs_foundation,
                "weight_decay": cfg.weight_decay_foundation,
                "max_length": cfg.max_length_foundation,
            },
            "fit_seconds": fit_seconds,
            "eval_metrics": eval_metrics,
            "test_metrics": test_metrics,
            "final_adapter_path": str(final_dir),
            "CUDA_device_memory_used": torch.cuda.memory_reserved(0), # get info about needed GPU memory
            "torch_version": torch.__version__,
        }

        # save metadata to json
    with open(run_dir / "metadata.json", "w") as f:
        json.dump(metadata, 
                  f, 
                  indent = 2)

    # free up some memory
    del model, base_model, trainer
    torch.cuda.empty_cache()

    print(f"[{name}] eval_f1={eval_metrics['f1_weighted']:.4f} fitted in {fit_seconds:.2f} seconds and was saved to {final_dir}")
    
    return metadata

We can now fit the models. \
This time we do not need to include `del model ...` in this cell because we already defined it in the `fit_and_save_qwen` function. At the encoder, we had to add it manually to training, because it was not defined inside the function (we forgot to do it).

In [34]:
results_foundation = {}

# fit ten models
for x in range(10):
    meta = fit_and_save_qwen(
        cfg,
        "Qwen2-05B-Foundation_" + str(x), # name each model after run
        X_train,
        y_train,
        X_test,
        y_test,
    )

    results_foundation[x] = meta
    

Loading weights: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 290/290 [00:00<00:00, 12700.99it/s]
[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Epoch,Training Loss,Validation Loss
1,0.210868,0.192309
2,0.163344,0.180840
3,0.142851,0.210064


[Qwen2-05B-Foundation_0] eval_f1=0.8580 fitted in 172.29 seconds and was saved to /home/simon/Documents/Machine learning Seminar/Models/Foundation Models Deterministic/Qwen2-05B-Foundation_0/final_adapter


Loading weights: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 290/290 [00:00<00:00, 19213.18it/s]
[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Epoch,Training Loss,Validation Loss
1,0.210868,0.192309
2,0.163344,0.180840


KeyboardInterrupt: 